# ViT vs ResNet on CIFAR-100: A Data Efficiency Study

**Hypothesis:** *Vision Transformers require more training data than CNNs (ResNet) to perform competitively on small datasets.*

This notebook compares a ResNet-18 (trained from scratch) against a pretrained ViT-Base (`google/vit-base-patch16-224-in21k`) on CIFAR-100 across varying training-set sizes (10%, 25%, 50%, 100%). We also compare inference speed and produce plots of accuracy, training time, and loss curves.

The notebook is designed to run on Google Colab with a GPU runtime and **does not require Google Drive access**.

### How to use this notebook

The training runs are **split into one cell per (model, fraction) combination** so you can run them one at a time, inspect the output, free memory between runs, or resume if a Colab session disconnects. Results are **persisted to a local pickle file** (`part3_results.pkl`) after every run so you never lose progress.

**Order of execution:**
1. Run the Setup, Section 1, Section 2, and Section 3 cells once at the start of each session.
2. Run the Section 4 *config* cell once — this creates (or reloads) the `results` dict.
3. Run any of the 8 training cells in Section 4 in any order. Each is independent and saves its output.
4. Once all 8 training cells have been run, run Section 5 (inference timing) and Section 6 (plots + summary).

## Setup

Install required packages, import libraries, set random seeds for reproducibility, and detect available hardware.

In [ ]:
# Install required packages (Colab)
!pip install -q transformers timm

In [ ]:
# Core imports
import os
import time
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision import models

import timm  # noqa: F401  (imported per assignment spec)
from transformers import ViTModel

# Reproducibility: fix seeds for torch, numpy, and Python's random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — training will be VERY slow on CPU.')

# Consistent colour scheme across plots
RESNET_COLOR = 'tab:blue'
VIT_COLOR = 'tab:orange'

## Section 1 — Data Loading

We load CIFAR-100 via `torchvision.datasets.CIFAR100`. Training images receive simple augmentation (random horizontal flip + random crop with padding). Both splits are normalised using the standard CIFAR-100 channel statistics.

Two DataLoaders are constructed here for the **ResNet** pipeline (32×32 native resolution). ViT-specific DataLoaders (224×224 resized) are built in Section 2.

In [ ]:
# CIFAR-100 channel statistics
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

# Training transforms: augmentation + normalisation (32x32 for ResNet)
train_transform_resnet = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Test transforms: just tensor conversion + normalisation
test_transform_resnet = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Download CIFAR-100 (stored under /content/data on Colab — no Drive needed)
DATA_ROOT = './data'
train_dataset_resnet = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=True, download=True, transform=train_transform_resnet
)
test_dataset_resnet = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=False, download=True, transform=test_transform_resnet
)

BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader_resnet = DataLoader(
    train_dataset_resnet, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
test_loader_resnet = DataLoader(
    test_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

print(f'Training set size: {len(train_dataset_resnet)}')
print(f'Test set size:     {len(test_dataset_resnet)}')
print(f'Number of classes: {len(train_dataset_resnet.classes)}')

## Section 2 — Model Setup

**ResNet-18** is loaded from `torchvision.models` with `pretrained=False` (trained from scratch) and its final FC layer replaced to output 100 classes.

**ViT-Base** is loaded pretrained from HuggingFace (`google/vit-base-patch16-224-in21k`). Since ViT expects 224×224 RGB inputs but CIFAR-100 is 32×32, we build **ViT-specific DataLoaders** that resize images to 224×224. A new linear classification head maps ViT's hidden size to 100 classes.

In [ ]:
def build_resnet():
    """Fresh ResNet-18 from scratch with a 100-way classifier."""
    model = models.resnet18(weights=None)  # pretrained=False equivalent
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model

# Build once just to report parameter count (we discard this instance)
_tmp_resnet = build_resnet()
resnet_params = sum(p.numel() for p in _tmp_resnet.parameters())
del _tmp_resnet
print(f'ResNet-18 total parameters: {resnet_params:,}')

In [ ]:
# ViT wrapper: HuggingFace ViTModel backbone + a linear classifier head
class ViTForCIFAR100(nn.Module):
    def __init__(self, pretrained_name='google/vit-base-patch16-224-in21k', num_classes=100):
        super().__init__()
        self.vit = ViTModel.from_pretrained(pretrained_name)
        hidden_size = self.vit.config.hidden_size
        # Classification head — applied to the [CLS] pooled output
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        pooled = outputs.pooler_output  # (B, hidden_size)
        return self.classifier(pooled)


def build_vit():
    """Pretrained ViT backbone with a freshly initialised classification head."""
    return ViTForCIFAR100()

# Build once just to report parameter count (we discard this instance)
_tmp_vit = build_vit()
vit_params = sum(p.numel() for p in _tmp_vit.parameters())
del _tmp_vit
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'ViT-Base total parameters: {vit_params:,}')

In [ ]:
# ViT expects 224x224 inputs — build separate datasets/loaders with a Resize transform.
train_transform_vit = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

test_transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_dataset_vit = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=True, download=True, transform=train_transform_vit
)
test_dataset_vit = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=False, download=True, transform=test_transform_vit
)

# ViT is memory-heavy on a T4 — default to 64 to avoid OOM. Bump up if you have an A100.
VIT_BATCH_SIZE = 64

test_loader_vit = DataLoader(
    test_dataset_vit, batch_size=VIT_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

print(f'ViT DataLoaders ready (224x224, batch size {VIT_BATCH_SIZE}).')

## Section 3 — Training Functions

Generic, model-agnostic helpers:
- `train_epoch` — one pass over the training set; returns avg loss and accuracy.
- `evaluate` — accuracy on a given loader.
- `train_model` — full training loop with Adam (lr=1e-4), CosineAnnealingLR scheduler, and CrossEntropyLoss. Tracks per-epoch metrics.

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """Run a single training epoch; return (avg_loss, accuracy_percent)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in dataloader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


def evaluate(model, dataloader, device):
    """Return accuracy (%) on the provided loader."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100.0 * correct / total

In [ ]:
def train_model(model, train_dataloader, test_dataloader, num_epochs, device, model_name):
    """Full training loop. Records per-epoch train/test acc, loss, and time."""
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()

    results = {
        'train_accuracies': [],
        'test_accuracies': [],
        'losses': [],
        'epoch_times': [],
    }

    print(f'\n=== Training {model_name} for {num_epochs} epochs ===')
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_dataloader, optimizer, criterion, device)
        test_acc = evaluate(model, test_dataloader, device)
        scheduler.step()
        epoch_time = time.time() - t0

        results['losses'].append(train_loss)
        results['train_accuracies'].append(train_acc)
        results['test_accuracies'].append(test_acc)
        results['epoch_times'].append(epoch_time)

        print(
            f'[{model_name}] Epoch {epoch:2d}/{num_epochs} | '
            f'loss={train_loss:.4f} | train_acc={train_acc:5.2f}% | '
            f'test_acc={test_acc:5.2f}% | time={epoch_time:.1f}s'
        )
    return results

## Section 4 — Main Experiment: Data Efficiency

**Hypothesis:** ViTs lack CNN-style inductive biases (locality, translation equivariance), so they should need **more data** to match CNN performance on small datasets. CIFAR-100 is small (50k training images) so we further subsample it.

For each fraction in `[0.1, 0.25, 0.5, 1.0]` we freshly initialise both models and train for 15 epochs. Note that ResNet starts from scratch while ViT uses ImageNet-21k pretrained weights (with a fresh head).

### How the cells in this section work

- The **config cell** below defines shared constants, the subset-loader helper, and persistence helpers. It also initialises (or reloads) the `results` dict from `part3_results.pkl`.
- The **8 training cells** that follow are fully independent — each one trains a single model at a single fraction and saves its result to disk. You can run them in any order, one at a time.
- If a Colab session dies mid-training, re-run Setup → Section 3 → the config cell, then run whichever training cells still need to be done.

In [ ]:
# ------------------------------------------------------------------
# Section 4 config — run this ONCE per Colab session before any
# training cells. It loads previously saved results from disk if
# they exist, so you can resume across sessions.
# ------------------------------------------------------------------

DATA_FRACTIONS = [0.1, 0.25, 0.5, 1.0]
NUM_EPOCHS = 15
RESULTS_PATH = 'part3_results.pkl'

def make_subset_loader(full_dataset, fraction, batch_size, num_workers=NUM_WORKERS, seed=SEED):
    """Deterministic random subset DataLoader. Same seed => same indices for ResNet & ViT."""
    n_total = len(full_dataset)
    n_keep = int(n_total * fraction)
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=g).tolist()
    indices = perm[:n_keep]
    subset = Subset(full_dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, 'rb') as f:
            return pickle.load(f)
    return {}

def save_results(r):
    with open(RESULTS_PATH, 'wb') as f:
        pickle.dump(r, f)

def record_run(fraction, model_key, run_results):
    """Persist one training run into the global results dict and write to disk."""
    results = load_results()
    if fraction not in results:
        results[fraction] = {}
    results[fraction][model_key] = {
        'accuracy': run_results['test_accuracies'][-1],
        'avg_epoch_time': float(np.mean(run_results['epoch_times'])),
        # Full curves saved too — needed for the 100% plots in Section 6
        'train_accuracies': run_results['train_accuracies'],
        'test_accuracies': run_results['test_accuracies'],
        'losses': run_results['losses'],
        'epoch_times': run_results['epoch_times'],
    }
    save_results(results)
    print(f'\nSaved results for fraction={fraction}, model={model_key} -> {RESULTS_PATH}')
    return results

# Load any results already persisted on disk
results = load_results()
print('Previously completed runs:')
if not results:
    print('  (none — start fresh)')
for frac in sorted(results.keys()):
    for mk, v in results[frac].items():
        print(f"  frac={frac}  model={mk}  final_acc={v['accuracy']:.2f}%  avg_epoch={v['avg_epoch_time']:.1f}s")

### ResNet-18 @ 10% data

In [ ]:
# ResNet-18 trained on 10% of CIFAR-100
FRAC = 0.1
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_loader = make_subset_loader(train_dataset_resnet, FRAC, batch_size=BATCH_SIZE)
model = build_resnet().to(device)
run = train_model(model, train_loader, test_loader_resnet, NUM_EPOCHS, device, f'ResNet18 @ {FRAC}')
results = record_run(FRAC, 'resnet', run)

# Free memory
del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ViT-Base @ 10% data

In [ ]:
# ViT-Base trained on 10% of CIFAR-100 (224x224 images)
FRAC = 0.1
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=VIT_BATCH_SIZE)
    model = build_vit().to(device)
    run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM — retrying with batch size 32. Consider reducing VIT_BATCH_SIZE.')
        torch.cuda.empty_cache()
        train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=32)
        model = build_vit().to(device)
        run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
    else:
        raise

results = record_run(FRAC, 'vit', run)

del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ResNet-18 @ 25% data

In [ ]:
FRAC = 0.25
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_loader = make_subset_loader(train_dataset_resnet, FRAC, batch_size=BATCH_SIZE)
model = build_resnet().to(device)
run = train_model(model, train_loader, test_loader_resnet, NUM_EPOCHS, device, f'ResNet18 @ {FRAC}')
results = record_run(FRAC, 'resnet', run)

del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ViT-Base @ 25% data

In [ ]:
FRAC = 0.25
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=VIT_BATCH_SIZE)
    model = build_vit().to(device)
    run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM — retrying with batch size 32.')
        torch.cuda.empty_cache()
        train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=32)
        model = build_vit().to(device)
        run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
    else:
        raise

results = record_run(FRAC, 'vit', run)

del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ResNet-18 @ 50% data

In [ ]:
FRAC = 0.5
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_loader = make_subset_loader(train_dataset_resnet, FRAC, batch_size=BATCH_SIZE)
model = build_resnet().to(device)
run = train_model(model, train_loader, test_loader_resnet, NUM_EPOCHS, device, f'ResNet18 @ {FRAC}')
results = record_run(FRAC, 'resnet', run)

del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ViT-Base @ 50% data

In [ ]:
FRAC = 0.5
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=VIT_BATCH_SIZE)
    model = build_vit().to(device)
    run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM — retrying with batch size 32.')
        torch.cuda.empty_cache()
        train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=32)
        model = build_vit().to(device)
        run = train_model(model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
    else:
        raise

results = record_run(FRAC, 'vit', run)

del model, run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ResNet-18 @ 100% data

This is the run whose trained weights are reused in Section 5 (inference timing), so we keep the model in a named variable `final_resnet_model`.

In [ ]:
FRAC = 1.0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_loader = make_subset_loader(train_dataset_resnet, FRAC, batch_size=BATCH_SIZE)
final_resnet_model = build_resnet().to(device)
run = train_model(final_resnet_model, train_loader, test_loader_resnet, NUM_EPOCHS, device, f'ResNet18 @ {FRAC}')
results = record_run(FRAC, 'resnet', run)

# Save ResNet weights too, so Section 5 can reload them even after a kernel restart
torch.save(final_resnet_model.state_dict(), 'resnet_final.pt')
print('Saved ResNet weights to resnet_final.pt')

del run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### ViT-Base @ 100% data

Longest single run of the notebook. Weights are cached to disk for reuse in Section 5.

In [ ]:
FRAC = 1.0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=VIT_BATCH_SIZE)
    final_vit_model = build_vit().to(device)
    run = train_model(final_vit_model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM — retrying with batch size 32.')
        torch.cuda.empty_cache()
        train_loader = make_subset_loader(train_dataset_vit, FRAC, batch_size=32)
        final_vit_model = build_vit().to(device)
        run = train_model(final_vit_model, train_loader, test_loader_vit, NUM_EPOCHS, device, f'ViT @ {FRAC}')
    else:
        raise

results = record_run(FRAC, 'vit', run)

torch.save(final_vit_model.state_dict(), 'vit_final.pt')
print('Saved ViT weights to vit_final.pt')

del run, train_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section 5 — Secondary Experiment: Inference Time

Using the models trained on 100% of the data, we measure wall-clock inference time on 1000 test images for each. This tells us not just who is *more accurate*, but who is *more efficient at prediction time* — an important practical consideration.

This cell will **reload saved weights** if `final_resnet_model` / `final_vit_model` aren't in memory (e.g. you ran the 100% training cells in a previous session).

In [ ]:
# Reload trained 100% models from disk if not already in memory
if 'final_resnet_model' not in globals() or final_resnet_model is None:
    final_resnet_model = build_resnet().to(device)
    final_resnet_model.load_state_dict(torch.load('resnet_final.pt', map_location=device))
    print('Loaded ResNet weights from resnet_final.pt')

if 'final_vit_model' not in globals() or final_vit_model is None:
    final_vit_model = build_vit().to(device)
    final_vit_model.load_state_dict(torch.load('vit_final.pt', map_location=device))
    print('Loaded ViT weights from vit_final.pt')


def measure_inference_time(model, dataset, n_images=1000, device=device):
    """Time inference on the first n_images of `dataset` (batch size 32)."""
    model.eval()
    indices = list(range(min(n_images, len(dataset))))
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

    # GPU warm-up
    if device.type == 'cuda':
        with torch.no_grad():
            sample, _ = next(iter(loader))
            _ = model(sample.to(device))
        torch.cuda.synchronize()

    t0 = time.time()
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device, non_blocking=True)
            _ = model(inputs)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    total_time = time.time() - t0

    return {
        'total_time': total_time,
        'time_per_image': total_time / n_images,
        'images_per_second': n_images / total_time,
        'n_images': n_images,
    }


print('Measuring ResNet inference time...')
resnet_inf = measure_inference_time(final_resnet_model, test_dataset_resnet, n_images=1000)
print('Measuring ViT inference time...')
vit_inf = measure_inference_time(final_vit_model, test_dataset_vit, n_images=1000)

# Persist inference timings so Section 6 can use them across sessions
with open('part3_inference.pkl', 'wb') as f:
    pickle.dump({'resnet': resnet_inf, 'vit': vit_inf}, f)

print('\n==================== Inference Timing ====================')
print(f"{'Model':<10}{'Total (s)':>12}{'Per image (ms)':>18}{'Images/sec':>15}")
print('-' * 55)
for name, inf in [('ResNet18', resnet_inf), ('ViT-Base', vit_inf)]:
    print(f"{name:<10}{inf['total_time']:>12.3f}{inf['time_per_image']*1000:>18.3f}{inf['images_per_second']:>15.1f}")
print('=' * 55)

## Section 6 — Results and Plots

We now visualise the findings:
1. **Data efficiency** — the headline result.
2. **Training time per epoch** — practical cost side of the story.
3. **Full training curves at 100% data** — train vs test accuracy over epochs.
4. **Loss curves at 100% data** — convergence behaviour.

All figures are rendered inline and saved to PNG files in the working directory. The cells read from the pickled results on disk, so they work even if you only ran the training cells in a prior Colab session.

In [ ]:
# Load persisted results
results = load_results()
assert results, 'No results found — run Section 4 training cells first.'

fractions_sorted = sorted(results.keys())
missing = [(f, m) for f in fractions_sorted for m in ('resnet', 'vit') if m not in results.get(f, {})]
if missing:
    print('WARNING: missing runs — plots will be partial until these are done:')
    for f, m in missing:
        print(f'  fraction={f}, model={m}')

resnet_accs  = [results[f]['resnet']['accuracy']       for f in fractions_sorted if 'resnet' in results[f]]
vit_accs     = [results[f]['vit']['accuracy']          for f in fractions_sorted if 'vit' in results[f]]
resnet_times = [results[f]['resnet']['avg_epoch_time'] for f in fractions_sorted if 'resnet' in results[f]]
vit_times    = [results[f]['vit']['avg_epoch_time']    for f in fractions_sorted if 'vit' in results[f]]
resnet_xs    = [f for f in fractions_sorted if 'resnet' in results[f]]
vit_xs       = [f for f in fractions_sorted if 'vit' in results[f]]

# --- Plot 1: Data Efficiency ---
plt.figure(figsize=(8, 5))
plt.plot(resnet_xs, resnet_accs, marker='o', color=RESNET_COLOR, label='ResNet-18')
plt.plot(vit_xs, vit_accs, marker='s', color=VIT_COLOR, label='ViT-Base')
plt.xlabel('Training data fraction')
plt.ylabel('Test accuracy (%)')
plt.title('Test Accuracy vs Training Data Size')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('data_efficiency.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 2: Training time per epoch vs data fraction ---
plt.figure(figsize=(8, 5))
plt.plot(resnet_xs, resnet_times, marker='o', color=RESNET_COLOR, label='ResNet-18')
plt.plot(vit_xs, vit_times, marker='s', color=VIT_COLOR, label='ViT-Base')
plt.xlabel('Training data fraction')
plt.ylabel('Average epoch time (s)')
plt.title('Training Time per Epoch vs Data Fraction')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('training_time.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 3: Full training curves at 100% data ---
assert 1.0 in results and 'resnet' in results[1.0] and 'vit' in results[1.0], \
    'Need both 100% runs completed to draw training curves.'
r = results[1.0]['resnet']
v = results[1.0]['vit']
epochs = list(range(1, len(r['train_accuracies']) + 1))

plt.figure(figsize=(9, 5.5))
plt.plot(epochs, r['train_accuracies'], color=RESNET_COLOR, linestyle='--', marker='o', label='ResNet train')
plt.plot(epochs, r['test_accuracies'],  color=RESNET_COLOR, linestyle='-',  marker='o', label='ResNet test')
plt.plot(epochs, v['train_accuracies'], color=VIT_COLOR,    linestyle='--', marker='s', label='ViT train')
plt.plot(epochs, v['test_accuracies'],  color=VIT_COLOR,    linestyle='-',  marker='s', label='ViT test')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Test Accuracy over Epochs — 100% Data')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 4: Loss curves at 100% data ---
plt.figure(figsize=(8, 5))
plt.plot(epochs, r['losses'], color=RESNET_COLOR, marker='o', label='ResNet-18')
plt.plot(epochs, v['losses'], color=VIT_COLOR,    marker='s', label='ViT-Base')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Training Loss over Epochs — 100% Data')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()

In [ ]:
# --- Final Summary Table ---

# Try to load cached inference timings if Section 5 was run in a previous session
if 'resnet_inf' not in globals() or 'vit_inf' not in globals():
    if os.path.exists('part3_inference.pkl'):
        with open('part3_inference.pkl', 'rb') as f:
            _inf = pickle.load(f)
        resnet_inf = _inf['resnet']
        vit_inf = _inf['vit']
    else:
        resnet_inf = vit_inf = None

print('\n' + '=' * 72)
print('FINAL SUMMARY: ViT vs ResNet on CIFAR-100')
print('=' * 72)
print(f"{'Fraction':>10}{'ResNet acc':>15}{'ViT acc':>12}{'Winner':>12}{'Gap (ViT-ResNet)':>22}")
print('-' * 72)
complete_fracs = [f for f in fractions_sorted if 'resnet' in results[f] and 'vit' in results[f]]
for f in complete_fracs:
    ra = results[f]['resnet']['accuracy']
    va = results[f]['vit']['accuracy']
    winner = 'ViT' if va > ra else ('ResNet' if ra > va else 'Tie')
    print(f"{f:>10.2f}{ra:>14.2f}%{va:>11.2f}%{winner:>12}{va - ra:>21.2f}%")
print('=' * 72)

print('\nParameter counts:')
print(f'  ResNet-18: {resnet_params:,}')
print(f'  ViT-Base:  {vit_params:,}')
print(f'  ViT has ~{vit_params / resnet_params:.1f}x more parameters than ResNet.')

if resnet_inf is not None and vit_inf is not None:
    print('\nInference time (1000 test images):')
    print(f"  ResNet-18: {resnet_inf['total_time']:.3f}s total | {resnet_inf['time_per_image']*1000:.2f} ms/img | {resnet_inf['images_per_second']:.1f} img/s")
    print(f"  ViT-Base:  {vit_inf['total_time']:.3f}s total | {vit_inf['time_per_image']*1000:.2f} ms/img | {vit_inf['images_per_second']:.1f} img/s")
else:
    print('\n(Inference timings not available — run Section 5.)')

# --- Conclusion ---
if len(complete_fracs) >= 2:
    low_frac = complete_fracs[0]
    high_frac = complete_fracs[-1]
    gap_low = results[low_frac]['vit']['accuracy'] - results[low_frac]['resnet']['accuracy']
    gap_high = results[high_frac]['vit']['accuracy'] - results[high_frac]['resnet']['accuracy']

    print('\n' + '-' * 72)
    print('CONCLUSION')
    print('-' * 72)
    print(
        'Hypothesis: "Vision Transformers require more training data than CNNs\n'
        '(ResNet) to perform competitively on small datasets."\n'
    )
    print(f'At {int(low_frac*100)}% data  : ViT - ResNet = {gap_low:+.2f}%')
    print(f'At {int(high_frac*100)}% data : ViT - ResNet = {gap_high:+.2f}%')

    if gap_high - gap_low > 0:
        verdict = (
            'The ViT/ResNet gap WIDENS in ViT\'s favour as more data is added,\n'
            'which is consistent with the hypothesis that ViTs benefit disproportionately\n'
            'from larger training sets.'
        )
    else:
        verdict = (
            'The ViT/ResNet gap does NOT widen in ViT\'s favour as more data is added.\n'
            'This does not straightforwardly support the hypothesis — note, however, that\n'
            'the ViT here is ImageNet-21k pretrained, which partially compensates for the\n'
            'data-hunger that the hypothesis describes.'
        )
    print('\n' + verdict)
    print('-' * 72)